In [47]:
from pathlib import Path
import pandas as pd

STATIC_PARAMETERS = {"RecordID", "Age", "Gender", "Height", "ICUType", "Weight"}
STATIC_EXCLUDE = {"RecordID", "Age", "Gender", "Height", "ICUType"}

def hhmm_to_minutes(value: str) -> int:
    hours, minutes = value.split(":")
    return int(hours) * 60 + int(minutes)

def parse_patient_file(file_path: Path):
    df = pd.read_csv(file_path)
    df["Value"] = pd.to_numeric(df["Value"], errors="coerce")

    record_rows = df.loc[df["Parameter"] == "RecordID", "Value"].dropna()
    patient_id = int(record_rows.iloc[0]) if not record_rows.empty else int(file_path.stem)

    static_df = df[df["Parameter"].isin(STATIC_PARAMETERS)].copy()
    static_df = static_df.drop_duplicates(subset=["Parameter"], keep="last")
    static_values = static_df.set_index("Parameter")["Value"].to_dict()

    static_record = {
        "RecordID": patient_id,
        "Age": static_values.get("Age"),
        "Gender": static_values.get("Gender"),
        "Height": static_values.get("Height"),
        "Weight": static_values.get("Weight"),
        "ICUType": static_values.get("ICUType"),
    }

    dynamic_df = df[~df["Parameter"].isin(STATIC_EXCLUDE)].copy()
    dynamic_df["Minutes"] = dynamic_df["Time"].map(hhmm_to_minutes)
    dynamic_df["RecordID"] = patient_id
    dynamic_df = dynamic_df[["RecordID", "Time", "Minutes", "Parameter", "Value"]]

    return static_record, dynamic_df

def load_cohort(folder: str):
    folder_path = Path(folder)
    static_records = []
    dynamic_tables = []

    for file_path in sorted(folder_path.glob("*.txt")):
        static_record, dynamic_df = parse_patient_file(file_path)
        static_records.append(static_record)
        dynamic_tables.append(dynamic_df)

    patients_static = pd.DataFrame(static_records).drop_duplicates(subset=["RecordID"])
    patients_static = patients_static.set_index("RecordID").sort_index()

    if dynamic_tables:
        patient_events = pd.concat(dynamic_tables, ignore_index=True)
        patient_events = patient_events.sort_values(["RecordID", "Minutes", "Parameter"]).reset_index(drop=True)
    else:
        patient_events = pd.DataFrame(columns=["RecordID", "Time", "Minutes", "Parameter", "Value"])

    return patients_static, patient_events

In [48]:
train_data_location = "set-a"
patients_static, patient_events = load_cohort(train_data_location)

print(f"Number of patients: {patients_static.shape[0]}")
print(f"Number of dynamic observations: {patient_events.shape[0]}")

display(patients_static.head())
display(patient_events.head(15))

Number of patients: 4000
Number of dynamic observations: 1737980


,Age,Gender,Height,Weight,ICUType
RecordID,,,,,
132539,54.0,0.0,-1.0,-1.0,4.0
132540,76.0,1.0,175.3,81.6,2.0
132541,44.0,0.0,-1.0,56.7,3.0
132543,68.0,1.0,180.3,84.6,3.0
132545,88.0,0.0,-1.0,-1.0,3.0


,RecordID,Time,Minutes,Parameter,Value
0,132539,00:00,0,Weight,-1.00
1,132539,00:07,7,GCS,15.00
2,132539,00:07,7,HR,73.00
3,132539,00:07,7,NIDiasABP,65.00
4,132539,00:07,7,NIMAP,92.33
5,132539,00:07,7,NISysABP,147.00
6,132539,00:07,7,RespRate,19.00
7,132539,00:07,7,Temp,35.10
8,132539,00:07,7,Urine,900.00
9,132539,00:37,37,HR,77.00


In [49]:
#create a pivot table of patient_events
pivot_table = patient_events.pivot_table(index=["RecordID", "Time", "Minutes"], columns="Parameter", values="Value")
print("Pivot table of patient_events:")
# display(pivot_table.head(5))
#find how many columns are in the pivot table
print(f"Number of columns in the pivot table: {pivot_table.shape[1]}")
#make it a dataframe again
pivot_table = pivot_table.reset_index()
print("Pivot table of patient_events after resetting index:")
# display(pivot_table.head(5))
#join the static variables to the pivot table
pivot_table = pivot_table.merge(patients_static, on="RecordID", how="left")
pivot_table = pivot_table.sort_values(["RecordID", "Minutes"]).reset_index(drop=True)
print("Pivot table of patient_events after merging with patients_static:")
# display(pivot_table.head(5))
#values that gender variable can take
print(f"Unique values in Gender column: {pivot_table['Gender'].unique()}")
#minus one for the 'Gender' 0,1,-1
print(f"Number of columns in the pivot table after merging with patients_static: {pivot_table.shape[1]}")
print(f"pivot table columns: {pivot_table.columns.tolist()}")

Pivot table of patient_events:
Number of columns in the pivot table: 37
Pivot table of patient_events after resetting index:
Pivot table of patient_events after merging with patients_static:
Unique values in Gender column: [ 0.  1. -1.]
Number of columns in the pivot table after merging with patients_static: 45
pivot table columns: ['RecordID', 'Time', 'Minutes', 'ALP', 'ALT', 'AST', 'Albumin', 'BUN', 'Bilirubin', 'Cholesterol', 'Creatinine', 'DiasABP', 'FiO2', 'GCS', 'Glucose', 'HCO3', 'HCT', 'HR', 'K', 'Lactate', 'MAP', 'MechVent', 'Mg', 'NIDiasABP', 'NIMAP', 'NISysABP', 'Na', 'PaCO2', 'PaO2', 'Platelets', 'RespRate', 'SaO2', 'SysABP', 'Temp', 'TroponinI', 'TroponinT', 'Urine', 'WBC', 'Weight_x', 'pH', 'Age', 'Gender', 'Height', 'Weight_y', 'ICUType']


In [50]:
pivot_table

,RecordID,Time,Minutes,ALP,ALT,AST,Albumin,BUN,Bilirubin,Cholesterol,...,TroponinT,Urine,WBC,Weight_x,pH,Age,Gender,Height,Weight_y,ICUType
0,132539,00:00,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,-1.0,NaN,54.0,0.0,-1.0,-1.0,4.0
1,132539,00:07,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,900.0,NaN,NaN,NaN,54.0,0.0,-1.0,-1.0,4.0
2,132539,00:37,37,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,60.0,NaN,NaN,NaN,54.0,0.0,-1.0,-1.0,4.0
3,132539,01:37,97,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,30.0,NaN,NaN,NaN,54.0,0.0,-1.0,-1.0,4.0
4,132539,02:37,157,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,170.0,NaN,NaN,NaN,54.0,0.0,-1.0,-1.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299259,142673,45:36,2736,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,23.0,NaN,87.3,NaN,78.0,0.0,157.5,87.3,4.0
299260,142673,45:39,2739,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,7.31,78.0,0.0,157.5,87.3,4.0
299261,142673,46:36,2796,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,40.0,NaN,87.3,NaN,78.0,0.0,157.5,87.3,4.0
299262,142673,47:21,2841,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,87.3,NaN,78.0,0.0,157.5,87.3,4.0


In [12]:
#find unique parameters in patient_events
unique_parameters = patient_events["Parameter"].unique()
print(f"Unique parameters in patient_events: {unique_parameters}")
print(f"length of unique parameters: {len(unique_parameters)}")
print(f"number of patients in patients_static: {patients_static.shape[0]}")

Unique parameters in patient_events: <StringArray>
[     'Weight',         'GCS',          'HR',   'NIDiasABP',       'NIMAP',
    'NISysABP',    'RespRate',        'Temp',       'Urine',         'HCT',
         'BUN',  'Creatinine',     'Glucose',        'HCO3',           'K',
          'Mg',          'Na',   'Platelets',         'WBC',       'PaCO2',
        'PaO2',          'pH',     'DiasABP',        'FiO2',         'MAP',
    'MechVent',      'SysABP',        'SaO2',         'ALP',         'ALT',
         'AST',     'Albumin',   'Bilirubin',     'Lactate', 'Cholesterol',
   'TroponinI',   'TroponinT']
Length: 37, dtype: str
length of unique parameters: 37
number of patients in patients_static: 4000


In [18]:
# load in outcomes for the training set (set-a)
outcomes = pd.read_csv("Outcomes-a.txt")
print(f"Number of outcome records: {outcomes.shape[0]}")
#number of null records
num_null_records = outcomes['In-hospital_death'].isnull().sum()
print(f"Number of null records in 'In-hospital_death': {num_null_records}")
outcomes#[['In-hospital_death']]

Number of outcome records: 4000
Number of null records in 'In-hospital_death': 0


,RecordID,SAPS-I,SOFA,Length_of_stay,Survival,In-hospital_death
0,132539,6,1,5,-1,0
1,132540,16,8,8,-1,0
2,132541,21,11,19,-1,0
3,132543,7,1,9,575,0
4,132545,17,2,4,918,0
...,...,...,...,...,...,...
3995,142665,19,7,10,336,0
3996,142667,8,2,3,-1,0
3997,142670,8,5,11,-1,0
3998,142671,22,10,8,7,1
